# Guide First Frame Review

这个 notebook 用来审阅 `Ideal centroid noise: truth detector + N(0, 0.065^2) pix + exact et_focalplane geometry` 的首帧导星联合解算结果。

这次统一采用两套明确分开的口径：
- `current_to_frame_truth`：当前解相对于“实际仿真帧真值”的姿态误差，这是当前最应该盯的主指标。
- `frame_truth_to_nominal_body`：实际仿真帧相对于名义 `et_focalplane/body` 几何的固定偏移，它主要反映 simulator 的 telescope FOV offset，而不是质心噪声。

建议阅读顺序：
1. 先看“推荐口径”和“高层结论”，确认这次该看哪组指标。
2. 再看“反事实姿态解算”和“telescope FOV offset 证据链”，理解误差来自哪里。
3. 最后看分探测器和逐星 dataframe，定位具体异常星或异常探测器。


## 1. 读取结果文件

这一节只做三件事：
- 读取主结果 JSON
- 读取详细误差审计 JSON
- 顺手读取首个 batch 的 `run_meta.json`，方便核对 `field_offset` 和 detector truth 口径


In [8]:
# 这格统一准备后面所有表格会用到的对象。
# 约定：
# - result: 主结果摘要 JSON
# - audit: 逐星详细审计 JSON
# - summary: audit 的全局统计摘要
# - counter: 反事实姿态解算对比块
# - per_star_df: 逐星 dataframe

from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")

ROOT = Path("/home/cxgao/ET/FSG/fsglib")
RESULT_PATH = ROOT / "outputs/debug/guide_first_frame_truth_noise_0065pix_exact_etcoord_result.json"
AUDIT_PATH = ROOT / "outputs/debug/guide_first_frame_truth_noise_0065pix_exact_etcoord_error_audit.json"

result = json.loads(RESULT_PATH.read_text(encoding="utf-8"))
audit = json.loads(AUDIT_PATH.read_text(encoding="utf-8"))
summary = result["error_audit"]["summary"]
counter = summary["counterfactual_solutions"]
per_star_df = pd.json_normalize(audit["per_star"])

dataset_root = Path(result["meta"]["dataset_root"])
first_detector = sorted(result["detector_stats"])[0]
first_batch_name = result["detector_stats"][first_detector]["batch_name"]
run_meta_path = dataset_root / first_batch_name / "run_meta.json"
run_meta = json.loads(run_meta_path.read_text(encoding="utf-8"))

print(f"Scenario   : Ideal centroid noise: truth detector + N(0, 0.065^2) pix + exact et_focalplane geometry")
print(f"Result JSON: {RESULT_PATH}")
print(f"Audit  JSON: {AUDIT_PATH}")
print(f"Run meta   : {run_meta_path}")
print(f"Per-star rows: {len(per_star_df)}")


Scenario   : Ideal centroid noise: truth detector + N(0, 0.065^2) pix + exact et_focalplane geometry
Result JSON: /home/cxgao/ET/FSG/fsglib/outputs/debug/guide_first_frame_truth_noise_0065pix_exact_etcoord_result.json
Audit  JSON: /home/cxgao/ET/FSG/fsglib/outputs/debug/guide_first_frame_truth_noise_0065pix_exact_etcoord_error_audit.json
Run meta   : /home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch3_ra305.0356_dec38.4755/run_meta.json
Per-star rows: 400


## 2. 推荐口径

这一步最重要，因为现在结果里同时有“当前帧真值口径”和“名义几何口径”。

简单记法：
- 真正汇报“算法在仿真帧上做得怎样”，优先看 `current_to_frame_truth`。
- 如果想解释为什么还存在一个稳定的几何偏差，再看 `frame_truth_to_nominal_body`。
- `predicted_vs_ecsv_detector` 近似为 0，说明 `et_focalplane` 的 raw detector 坐标本身是对的。
- `predicted_vs_truth` 的常量偏移来自 NPZ detector truth 里额外带入的 telescope FOV offset。


In [9]:
# 这张表是 notebook 里最先该统一认知的一张表。
metric_definition_df = pd.DataFrame([
    {
        "metric": "current_to_frame_truth_arcsec",
        "recommended": True,
        "meaning": "当前解相对于实际仿真帧真值的姿态误差，当前最推荐的主指标。",
    },
    {
        "metric": "frame_truth_to_nominal_body_arcsec",
        "recommended": True,
        "meaning": "实际仿真帧相对于名义 et_focalplane/body 几何的固定偏移，主要反映 simulator telescope FOV offset。",
    },
    {
        "metric": "match_predicted_vs_ecsv_detector_pix",
        "recommended": True,
        "meaning": "参考星预测像点相对于 raw detector 坐标的偏差；若接近 0，说明 et_focalplane raw detector 几何是自洽的。",
    },
    {
        "metric": "match_predicted_vs_truth_pix",
        "recommended": False,
        "meaning": "参考星预测像点相对于 NPZ detector truth 的偏差；当前数据里它会混入 telescope FOV offset。",
    },
    {
        "metric": "delta_current_to_oracle_arcsec",
        "recommended": False,
        "meaning": "历史口径，当前不再作为首选主指标，因为它会把 frame truth 和 nominal body 混在一起。",
    },
])
display(metric_definition_df)

interpretation_df = pd.DataFrame(
    [{"key": key, "meaning": value} for key, value in summary.get("interpretation", {}).items()]
)
display(interpretation_df)


,metric,recommended,meaning
0,current_to_frame_truth_arcsec,True,当前解相对于实际仿真帧真值的姿态误差，当前最推荐的主指标。
1,frame_truth_to_nominal_body_arcsec,True,实际仿真帧相对于名义 et_focalplane/body 几何的固定偏移，主要反映 simulator telescope FOV offset。
2,match_predicted_vs_ecsv_detector_pix,True,参考星预测像点相对于 raw detector 坐标的偏差；若接近 0，说明 et_focalplane raw detector 几何是自洽的。
3,match_predicted_vs_truth_pix,False,参考星预测像点相对于 NPZ detector truth 的偏差；当前数据里它会混入 telescope FOV offset。
4,delta_current_to_oracle_arcsec,False,历史口径，当前不再作为首选主指标，因为它会把 frame truth 和 nominal body 混在一起。


,key,meaning
0,match_predicted_vs_truth_pix,Reference predicted_xy compared against NPZ detector truth. In this dataset NPZ detector truth includes the simulator telescope FOV offset.
1,match_predicted_vs_ecsv_detector_pix,Reference predicted_xy compared against raw et_focalplane detector coordinates from stars.ecsv.
2,frame_truth_reference,Use counterfactual_solutions.frame_truth_same_matches and delta_current_to_frame_truth_arcsec as the primary simulated-frame attitude accuracy metric.


## 3. 高层结论

这一节把最关键的数压缩成一张表：
- 姿态是否有效
- 残差 RMS / MAX
- 质心 RMS
- 当前解到“实际仿真帧真值”的姿态差
- 实际仿真帧到“名义几何”的固定偏移
- raw detector 与 NPZ detector truth 的两套坐标差异

这张表最适合直接截图或放进报告首页。


In [10]:
# 这里把最关键的指标都收敛到一张表里。
current_to_frame_truth = counter["delta_components"]["current_to_frame_truth"]
frame_truth_to_nominal = counter["delta_components"]["frame_truth_to_nominal_body"]

overview = pd.DataFrame([
    {"section": "scenario", "metric": "label", "value": "Ideal centroid noise: truth detector + N(0, 0.065^2) pix + exact et_focalplane geometry"},
    {"section": "attitude", "metric": "valid", "value": result["solution"]["valid"]},
    {"section": "attitude", "metric": "num_matched", "value": result["solution"]["num_matched"]},
    {"section": "attitude", "metric": "residual_rms_arcsec", "value": result["solution"]["residual_rms_arcsec"]},
    {"section": "attitude", "metric": "residual_max_arcsec", "value": result["solution"]["residual_max_arcsec"]},
    {"section": "matching", "metric": "mean_residual_pix", "value": result["matching"]["debug"]["mean_residual_pix"]},
    {"section": "centroid", "metric": "detector_rms_pix", "value": summary["centroid_error_detector_pix"]["rms_radial"]},
    {"section": "body", "metric": "body_total_rms_arcsec", "value": summary["body_error_total_arcsec"]["rms"]},
    {"section": "recommended", "metric": "current_to_frame_truth_total_arcsec", "value": current_to_frame_truth["total_arcsec"]},
    {"section": "recommended", "metric": "current_to_frame_truth_non_roll_arcsec", "value": current_to_frame_truth["non_roll_arcsec"]},
    {"section": "recommended", "metric": "current_to_frame_truth_roll_arcsec", "value": current_to_frame_truth["roll_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_total_arcsec", "value": frame_truth_to_nominal["total_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_non_roll_arcsec", "value": frame_truth_to_nominal["non_roll_arcsec"]},
    {"section": "nominal_gap", "metric": "frame_truth_to_nominal_roll_arcsec", "value": frame_truth_to_nominal["roll_arcsec"]},
    {"section": "detector_coords", "metric": "predicted_vs_raw_detector_rms_pix", "value": summary["match_predicted_vs_ecsv_detector_pix"]["rms_radial"]},
    {"section": "detector_coords", "metric": "npz_minus_raw_detector_offset_rms_pix", "value": summary["npz_minus_ecsv_detector_offset_pix"]["rms_radial"]},
])
display(overview)

quaternion_df = pd.DataFrame([
    {
        "q_w": result["solution"]["q_ib"][0],
        "q_x": result["solution"]["q_ib"][1],
        "q_y": result["solution"]["q_ib"][2],
        "q_z": result["solution"]["q_ib"][3],
    }
])
display(quaternion_df)


,section,metric,value
0,scenario,label,"Ideal centroid noise: truth detector + N(0, 0.065^2) pix + exact et_focalplane geometry"
1,attitude,valid,True
2,attitude,num_matched,400
3,attitude,residual_rms_arcsec,0.262772
4,attitude,residual_max_arcsec,0.682800
5,matching,mean_residual_pix,0.151493
6,centroid,detector_rms_pix,0.088031
7,body,body_total_rms_arcsec,0.488261
8,recommended,current_to_frame_truth_total_arcsec,0.028477
9,recommended,current_to_frame_truth_non_roll_arcsec,0.017922


,q_w,q_x,q_y,q_z
0,0.685905,-0.320306,0.130820,-0.640175


## 4. 反事实姿态解算

这一节是误差传播的主表。

四套解的推荐理解：
- `current`：当前真实链路结果
- `frame_truth_same_matches`：把观测 LOS 换成“实际仿真帧真值”，但保留当前匹配
- `nominal_body_same_matches`：把观测 LOS 换成名义 `et_focalplane/body` 几何
- `oracle_truth_body_and_match`：真值 body + 真值匹配，作为极限参考

在 exact 几何链路下：
- `current -> frame_truth` 更像真实算法误差
- `frame_truth -> nominal_body` 更像 simulator 与 nominal geometry 的口径差


In [11]:
# 先看四套姿态解本身的质量指标。
name_map = {
    "current": "current",
    "frame_truth_same_matches": "frame_truth_same_matches",
    "truth_pixel_same_matches": "truth_pixel_same_matches (legacy alias)",
    "nominal_body_same_matches": "nominal_body_same_matches",
    "exact_body_same_matches": "exact_body_same_matches (legacy alias)",
    "oracle_truth_body_and_match": "oracle_truth_body_and_match",
}

counter_rows = []
for key in [
    "current",
    "frame_truth_same_matches",
    "nominal_body_same_matches",
    "oracle_truth_body_and_match",
]:
    payload = counter.get(key)
    if payload is None:
        continue
    counter_rows.append({
        "solution_name": name_map.get(key, key),
        "valid": payload["valid"],
        "num_matched": payload["num_matched"],
        "num_rejected": payload["num_rejected"],
        "residual_rms_arcsec": payload["residual_rms_arcsec"],
        "residual_max_arcsec": payload["residual_max_arcsec"],
        "quality_flag": payload["quality_flag"],
        "degraded_level": payload["degraded_level"],
    })

display(pd.DataFrame(counter_rows))


,solution_name,valid,num_matched,num_rejected,residual_rms_arcsec,residual_max_arcsec,quality_flag,degraded_level
0,current,True,400,0,0.262772,0.682800,VALID,NORMAL_4D
1,frame_truth_same_matches,True,400,0,0.009935,0.013745,VALID,NORMAL_4D
2,nominal_body_same_matches,True,400,0,0.000972,0.004347,VALID,NORMAL_4D
3,oracle_truth_body_and_match,True,400,0,0.000972,0.004347,VALID,NORMAL_4D


In [12]:
# 再看姿态差如何分解成 total / non-roll / roll。
delta_components_df = pd.DataFrame([
    {"delta_name": key, **value}
    for key, value in counter["delta_components"].items()
])
display(delta_components_df)


,delta_name,non_roll_arcsec,roll_arcsec,total_arcsec
0,current_to_frame_truth,0.017922,0.022212,0.028477
1,current_to_truth_pixel,0.017922,0.022212,0.028477
2,frame_truth_to_nominal_body,0.432765,0.007098,0.432841
3,truth_pixel_to_exact_body,0.432765,0.007098,0.432841
4,current_to_nominal_body,0.414991,0.015114,0.415273
5,current_to_exact_body,0.414991,0.015114,0.415273
6,current_to_oracle,0.414991,0.015114,0.415273
7,exact_body_to_oracle_match,0.000000,0.000000,0.000000


## 5. telescope FOV offset 证据链

这里专门回答一个容易混淆的问题：

为什么 `run_meta` 里脚本层 `field_offset_x/y = 0`，但 `NPZ detector truth` 和 `stars.ecsv raw detector` 之间仍然差了一个固定常量？

当前证据链是：
1. `run_meta` 只记录脚本层 static field offset，并没有写 telescope 级随机偏移。
2. `predicted_vs_raw_detector` 近似 0，说明 `et_focalplane` raw detector 坐标没问题。
3. `predicted_vs_truth` 与 `npz_minus_raw_detector_offset` 相同，说明这个偏移来自 NPZ detector truth 的定义，而不是匹配器。


In [13]:
# 先看 run_meta 里能直接读到的 offset 相关字段。
run_meta_offset_df = pd.DataFrame([{
    "apply_static_field_offset": run_meta.get("apply_static_field_offset"),
    "field_offset_x_pix": run_meta.get("field_offset_x_pix"),
    "field_offset_y_pix": run_meta.get("field_offset_y_pix"),
    "requested_field_offset_x_pix": run_meta.get("requested_field_offset_x_pix"),
    "requested_field_offset_y_pix": run_meta.get("requested_field_offset_y_pix"),
    "guide_query_target_center_xpix": run_meta.get("guide_query_target_center_xpix"),
    "guide_query_target_center_ypix": run_meta.get("guide_query_target_center_ypix"),
}])
display(run_meta_offset_df)

# 再看这次真正影响理解的几项证据。
evidence_df = pd.DataFrame([
    {"metric": "sim_to_detector_map_rms_pix", "value": summary["sim_to_detector_map_error_pix"]["rms_radial"]},
    {"metric": "predicted_vs_raw_detector_rms_pix", "value": summary["match_predicted_vs_ecsv_detector_pix"]["rms_radial"]},
    {"metric": "predicted_vs_npz_truth_rms_pix", "value": summary["match_predicted_vs_truth_pix"]["rms_radial"]},
    {"metric": "npz_minus_raw_detector_offset_rms_pix", "value": summary["npz_minus_ecsv_detector_offset_pix"]["rms_radial"]},
])
display(evidence_df)

per_detector_offset_rows = []
for detector_id, payload in audit["per_detector"].items():
    per_detector_offset_rows.append({
        "detector_id": detector_id,
        "predicted_vs_raw_detector_rms_pix": payload["match_predicted_vs_ecsv_detector_pix"]["rms_radial"],
        "predicted_vs_npz_truth_rms_pix": payload["match_predicted_vs_truth_pix"]["rms_radial"],
        "npz_minus_raw_detector_offset_rms_pix": payload["npz_minus_ecsv_detector_offset_pix"]["rms_radial"],
        "npz_minus_raw_detector_mean_dx_pix": payload["npz_minus_ecsv_detector_offset_pix"]["mean_dx"],
        "npz_minus_raw_detector_mean_dy_pix": payload["npz_minus_ecsv_detector_offset_pix"]["mean_dy"],
    })
display(pd.DataFrame(per_detector_offset_rows).sort_values("detector_id"))


,apply_static_field_offset,field_offset_x_pix,field_offset_y_pix,requested_field_offset_x_pix,requested_field_offset_y_pix,guide_query_target_center_xpix,guide_query_target_center_ypix
0,False,0.000000,0.000000,None,None,1024.849292,1024.583885


,metric,value
0,sim_to_detector_map_rms_pix,0.000000
1,predicted_vs_raw_detector_rms_pix,0.000000
2,predicted_vs_npz_truth_rms_pix,0.141759
3,npz_minus_raw_detector_offset_rms_pix,0.141759


,detector_id,predicted_vs_raw_detector_rms_pix,predicted_vs_npz_truth_rms_pix,npz_minus_raw_detector_offset_rms_pix,npz_minus_raw_detector_mean_dx_pix,npz_minus_raw_detector_mean_dy_pix
3,guide_bottom,0.000000,0.141759,0.141759,0.132633,-0.050039
0,guide_left,0.000000,0.141759,0.141759,0.132633,-0.050039
2,guide_right,0.000000,0.141759,0.141759,0.132633,-0.050039
1,guide_top,0.000000,0.141759,0.141759,0.132633,-0.050039


## 6. 分探测器链路通过情况

这一节先只看数量：
- 每片 detector 选了多少星
- 最终匹配了多少星
- 参考星有多少

如果这里已经不通，后面的精度表就没有解释意义。


In [14]:
detector_counts_rows = []
for detector_id, stats in result["detector_stats"].items():
    row = {"detector_id": detector_id, **stats}
    detector_counts_rows.append(row)

detector_counts_df = pd.DataFrame(detector_counts_rows).sort_values("detector_id")
display(detector_counts_df)


,detector_id,batch_name,frame_path,num_truth_stars_visible,num_candidates_raw,num_candidates_selected,selection_mode,noise_mean_pix,noise_sigma_pix,random_seed,num_matched,num_reference_stars,sim_to_detector_kind,schema_version,offset_x_pix,offset_y_pix
3,guide_bottom,batch3_ra305.0356_dec38.4755,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch3_ra305.0356_dec38.4755/frames/scope0_coadd_000000_000000.npz,200,200,100,truth_brightness_proxy,0.000000,0.065000,20260326,100,260,offset,2,51.849292,51.583885
0,guide_left,batch0_ra278.1844_dec37.5704,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch0_ra278.1844_dec37.5704/frames/scope0_coadd_000000_000000.npz,184,184,100,truth_brightness_proxy,0.000000,0.065000,20260326,100,260,offset,2,51.592127,51.384965
2,guide_right,batch2_ra310.6239_dec59.2676,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch2_ra310.6239_dec59.2676/frames/scope0_coadd_000000_000000.npz,169,169,100,truth_brightness_proxy,0.000000,0.065000,20260326,100,260,offset,2,50.108021,51.848764
1,guide_top,batch1_ra269.5821_dec57.8997,/home/cxgao/ET/FSG_guide_sims/guide_det_v1_noise_psf_6s/batch1_ra269.5821_dec57.8997/frames/scope0_coadd_000000_000000.npz,200,200,100,truth_brightness_proxy,0.000000,0.065000,20260326,100,220,offset,2,51.380730,50.110761


## 7. 分探测器关键精度

这一节横向比较四片导星的关键误差：
- 质心误差
- raw detector / NPZ detector truth 的两套口径差异
- `body` 方向误差
- 像面匹配残差

如果某片明显异常，通常从这里最容易先看出来。


In [15]:
detector_metric_rows = []
for detector_id, payload in audit["per_detector"].items():
    detector_metric_rows.append({
        "detector_id": detector_id,
        "centroid_rms_pix": payload["centroid_error_detector_pix"]["rms_radial"],
        "body_centroid_rms_arcsec": payload["body_error_centroid_arcsec"]["rms"],
        "body_geometry_rms_arcsec": payload["body_error_geometry_arcsec"]["rms"],
        "body_total_rms_arcsec": payload["body_error_total_arcsec"]["rms"],
        "match_obs_pred_rms_pix": payload["match_observed_vs_predicted_pix"]["rms"],
        "predicted_vs_raw_detector_rms_pix": payload["match_predicted_vs_ecsv_detector_pix"]["rms_radial"],
        "predicted_vs_npz_truth_rms_pix": payload["match_predicted_vs_truth_pix"]["rms_radial"],
        "npz_minus_raw_detector_offset_rms_pix": payload["npz_minus_ecsv_detector_offset_pix"]["rms_radial"],
        "solution_model_rms_arcsec": payload["solution_residual_model_arcsec"]["rms"],
        "solution_exact_body_rms_arcsec": payload["solution_residual_exact_body_arcsec"]["rms"],
    })

detector_metric_df = pd.DataFrame(detector_metric_rows).sort_values("detector_id")
display(detector_metric_df)


,detector_id,centroid_rms_pix,body_centroid_rms_arcsec,body_geometry_rms_arcsec,body_total_rms_arcsec,match_obs_pred_rms_pix,predicted_vs_raw_detector_rms_pix,predicted_vs_npz_truth_rms_pix,npz_minus_raw_detector_offset_rms_pix,solution_model_rms_arcsec,solution_exact_body_rms_arcsec
3,guide_bottom,0.093009,0.278851,0.427784,0.467075,0.155177,0.000000,0.141759,0.141759,0.275058,0.397044
0,guide_left,0.084977,0.254580,0.420948,0.509045,0.171048,0.000000,0.141759,0.141759,0.255175,0.414683
2,guide_right,0.083305,0.249574,0.422858,0.489547,0.163910,0.000000,0.141759,0.141759,0.250019,0.415165
1,guide_top,0.090480,0.271484,0.428832,0.486471,0.161256,0.000000,0.141759,0.141759,0.270031,0.404763


## 8. 逐星 dataframe

这一节把详细审计 JSON 展平成 `per_star_df`。
后面如果要抓最差的星、检查错配、看单颗星误差传播，都从这张表开始。


In [16]:
print("per_star_df shape:", per_star_df.shape)

focus_cols = [
    "detector_id",
    "observed_source_id",
    "truth_index",
    "snr",
    "centroid_error_detector_radial_pix",
    "body_error_total_arcsec",
    "match_residual_pix",
    "predicted_vs_ecsv_detector_radial_pix",
    "predicted_vs_truth_radial_pix",
    "match_is_correct",
]
display(per_star_df[focus_cols].head(10))


per_star_df shape: (400, 76)


,detector_id,observed_source_id,truth_index,snr,centroid_error_detector_radial_pix,body_error_total_arcsec,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,match_is_correct
0,guide_left,guide_left:1,1,1088.537760,0.036234,0.376649,0.126947,0.000000,0.141759,True
1,guide_left,guide_left:2,2,617.013064,0.092934,0.595031,0.200071,0.000000,0.141759,True
2,guide_left,guide_left:3,3,556.357135,0.058597,0.412868,0.138141,0.000000,0.141759,True
3,guide_left,guide_left:4,4,440.411130,0.113695,0.743153,0.250170,0.000000,0.141759,True
4,guide_left,guide_left:5,5,412.145106,0.021198,0.460864,0.155324,0.000000,0.141759,True
5,guide_left,guide_left:6,6,407.012190,0.042827,0.516344,0.173967,0.000000,0.141759,True
6,guide_left,guide_left:7,7,369.621453,0.059750,0.497684,0.167791,0.000000,0.141759,True
7,guide_left,guide_left:8,8,352.483931,0.074468,0.362279,0.121723,0.000000,0.141759,True
8,guide_left,guide_left:9,9,341.991937,0.062903,0.393323,0.132038,0.000000,0.141759,True
9,guide_left,guide_left:10,10,330.635087,0.071389,0.629395,0.211323,0.000000,0.141759,True


## 9. 最大误差星表

下面几张表分别抓出：
- 质心最差的星
- `body` 总误差最大的星
- 像面匹配残差最大的星
- detector truth 口径差最显著的星


In [17]:
centroid_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "centroid_error_detector_radial_pix", "body_error_centroid_arcsec", "body_error_total_arcsec",
]
body_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "body_error_centroid_arcsec", "body_error_geometry_arcsec", "body_error_total_arcsec",
    "solution_residual_model_arcsec", "solution_residual_exact_body_arcsec",
]
match_cols = [
    "detector_id", "observed_source_id", "truth_index", "snr",
    "match_residual_pix", "predicted_vs_ecsv_detector_radial_pix", "predicted_vs_truth_radial_pix",
    "matched_catalog_id", "truth_catalog_source_id", "match_is_correct",
]

display(per_star_df[centroid_cols].sort_values("centroid_error_detector_radial_pix", ascending=False).head(20))
display(per_star_df[body_cols].sort_values("body_error_total_arcsec", ascending=False).head(20))
display(per_star_df[match_cols].sort_values("match_residual_pix", ascending=False).head(20))
display(per_star_df[match_cols].sort_values("predicted_vs_truth_radial_pix", ascending=False).head(20))


,detector_id,observed_source_id,truth_index,snr,centroid_error_detector_radial_pix,body_error_centroid_arcsec,body_error_total_arcsec
365,guide_bottom,guide_bottom:66,66,249.681769,0.221463,0.661757,0.982654
148,guide_top,guide_top:48,48,114.940694,0.194663,0.585734,0.990296
160,guide_top,guide_top:60,60,103.366101,0.189042,0.568189,0.397682
273,guide_right,guide_right:74,74,131.635066,0.186277,0.562693,0.839888
124,guide_top,guide_top:24,24,147.874415,0.185524,0.552878,0.763147
338,guide_bottom,guide_bottom:39,39,305.084325,0.184453,0.548280,0.785919
86,guide_left,guide_left:86,86,116.623026,0.183899,0.553698,0.873407
357,guide_bottom,guide_bottom:58,58,264.505782,0.178403,0.534521,0.883622
28,guide_left,guide_left:29,29,188.182709,0.172698,0.519545,0.474906
119,guide_top,guide_top:20,20,155.219892,0.170300,0.511703,0.355634


,detector_id,observed_source_id,truth_index,snr,body_error_centroid_arcsec,body_error_geometry_arcsec,body_error_total_arcsec,solution_residual_model_arcsec,solution_residual_exact_body_arcsec
148,guide_top,guide_top:48,48,114.940694,0.585734,0.429379,0.990296,0.606424,0.404699
365,guide_bottom,guide_bottom:66,66,249.681769,0.661757,0.426243,0.982654,0.682800,0.395658
257,guide_right,guide_right:58,58,141.914250,0.492561,0.423162,0.913196,0.500032,0.415139
383,guide_bottom,guide_bottom:84,84,204.674184,0.499966,0.428740,0.908602,0.529112,0.397741
399,guide_bottom,guide_bottom:100,100,182.572342,0.492158,0.428520,0.883932,0.518899,0.397646
357,guide_bottom,guide_bottom:58,58,264.505782,0.534521,0.428134,0.883622,0.554695,0.397266
86,guide_left,guide_left:86,86,116.623026,0.553698,0.421709,0.873407,0.560860,0.414740
376,guide_bottom,guide_bottom:77,77,217.081318,0.486171,0.426587,0.866680,0.512008,0.396361
289,guide_right,guide_right:90,90,118.818588,0.434170,0.423051,0.856599,0.441763,0.415162
212,guide_right,guide_right:13,13,299.802120,0.438533,0.422470,0.845945,0.444417,0.415253


,detector_id,observed_source_id,truth_index,snr,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,matched_catalog_id,truth_catalog_source_id,match_is_correct
148,guide_top,guide_top:48,48,114.940694,0.328129,0.000000,0.141759,1421699497135072000,1421699497135072000,True
365,guide_bottom,guide_bottom:66,66,249.681769,0.327518,0.000000,0.141759,2061022009267799808,2061022009267799808,True
257,guide_right,guide_right:58,58,141.914250,0.306112,0.000000,0.141759,2193584010190977024,2193584010190977024,True
383,guide_bottom,guide_bottom:84,84,204.674184,0.300584,0.000000,0.141759,2061299051851594880,2061299051851594880,True
357,guide_bottom,guide_bottom:58,58,264.505782,0.293356,0.000000,0.141759,2061286540593527808,2061286540593527808,True
399,guide_bottom,guide_bottom:100,100,182.572342,0.292767,0.000000,0.141759,2060934907346248192,2060934907346248192,True
86,guide_left,guide_left:86,86,116.623026,0.291985,0.000000,0.141759,2096367116205759744,2096367116205759744,True
376,guide_bottom,guide_bottom:77,77,217.081318,0.287967,0.000000,0.141759,2061126840827421312,2061126840827421312,True
289,guide_right,guide_right:90,90,118.818588,0.287283,0.000000,0.141759,2193585453300017792,2193585453300017792,True
212,guide_right,guide_right:13,13,299.802120,0.283624,0.000000,0.141759,2193763952138874624,2193763952138874624,True


,detector_id,observed_source_id,truth_index,snr,match_residual_pix,predicted_vs_ecsv_detector_radial_pix,predicted_vs_truth_radial_pix,matched_catalog_id,truth_catalog_source_id,match_is_correct
356,guide_bottom,guide_bottom:57,57,265.702013,0.088824,0.000000,0.141759,2061275579830803584,2061275579830803584,True
284,guide_right,guide_right:85,85,120.833435,0.144797,0.000000,0.141759,2193485260301842432,2193485260301842432,True
292,guide_right,guide_right:93,93,117.673705,0.159951,0.000000,0.141759,2193493231756660864,2193493231756660864,True
90,guide_left,guide_left:90,90,114.076685,0.123147,0.000000,0.141759,2108412712764502528,2108412712764502528,True
37,guide_left,guide_left:38,38,168.693725,0.203723,0.000000,0.141759,2096104779603933440,2096104779603933440,True
300,guide_bottom,guide_bottom:1,1,1961.475547,0.141992,0.000000,0.141759,2061242908036996352,2061242908036996352,True
123,guide_top,guide_top:23,23,147.880854,0.147793,0.000000,0.141759,1421877510643827840,1421877510643827840,True
349,guide_bottom,guide_bottom:50,50,286.678120,0.217092,0.000000,0.141759,2061415909308447488,2061415909308447488,True
205,guide_right,guide_right:6,6,497.814467,0.093366,0.000000,0.141759,2193638745252745856,2193638745252745856,True
396,guide_bottom,guide_bottom:97,97,183.651151,0.117504,0.000000,0.141759,2061080863197553024,2061080863197553024,True


## 10. 统计分布

最后一节给全局统计和按 detector 分组统计。
如果你想快速判断“是不是存在明显系统偏差”，这一节最有用。


In [18]:
numeric_cols = [
    "centroid_error_detector_radial_pix",
    "body_error_centroid_arcsec",
    "body_error_geometry_arcsec",
    "body_error_total_arcsec",
    "match_residual_pix",
    "predicted_vs_ecsv_detector_radial_pix",
    "predicted_vs_truth_radial_pix",
    "solution_residual_model_arcsec",
    "solution_residual_exact_body_arcsec",
]

display(per_star_df[numeric_cols].describe().T)
grouped = per_star_df.groupby("detector_id")[numeric_cols].agg(["mean", "median", "max"])
display(grouped)


,count,mean,std,min,25%,50%,75%,max
centroid_error_detector_radial_pix,400.000000,0.077215,0.042331,0.004436,0.045129,0.071702,0.106761,0.221463
body_error_centroid_arcsec,400.000000,0.231420,0.126984,0.013040,0.135028,0.215664,0.320299,0.661757
body_error_geometry_arcsec,400.000000,0.425105,0.003423,0.419114,0.421804,0.425711,0.428225,0.430796
body_error_total_arcsec,400.000000,0.453977,0.179960,0.032089,0.326973,0.428046,0.572745,0.990296
match_residual_pix,400.000000,0.151493,0.060091,0.010569,0.108819,0.142157,0.190508,0.328129
predicted_vs_ecsv_detector_radial_pix,400.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
predicted_vs_truth_radial_pix,400.000000,0.141759,0.000000,0.141759,0.141759,0.141759,0.141759,0.141759
solution_residual_model_arcsec,400.000000,0.229978,0.127279,0.016552,0.134212,0.213542,0.313161,0.682800
solution_residual_exact_body_arcsec,400.000000,0.407913,0.007563,0.394785,0.402122,0.410307,0.415014,0.415264


centroid_error_detector_radial_pix                   body_error_centroid_arcsec                   body_error_geometry_arcsec                   body_error_total_arcsec                   match_residual_pix  \
                                           mean   median      max                       mean   median      max                       mean   median      max                    mean   median      max               mean   
detector_id                                                                                                                                                                                                                
guide_bottom                           0.080724 0.075669 0.221463                   0.242000 0.227912 0.661757                   0.427783 0.427654 0.429731                0.430648 0.403672 0.982654           0.143126   
guide_left                             0.076703 0.072643 0.183899                   0.229769 0.217952 0.553698                   0.420947 0.420890 0.423374                0.478578 0.470365 0.873407           0.160784   
guide_right                            0.073226 0.068825 0.186277                   0.219386 0.206000 0.562693                   0.422857 0.422816 0.425633                0.454936 0.434977 0.913196           0.152272   
guide_top                              0.078206 0.073018 0.194663                   0.234526 0.218004 0.585734                   0.428831 0.428713 0.430796                0.451745 0.425812 0.990296           0.149789   

                               predicted_vs_ecsv_detector_radial_pix                   predicted_vs_truth_radial_pix                   solution_residual_model_arcsec                    \
               median      max                                  mean   median      max                          mean   median      max                           mean   median      max   
detector_id                                                                                                                                                                               
guide_bottom 0.134690 0.327518                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       0.237319 0.214633 0.682800   
guide_left   0.157229 0.291985                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       0.230480 0.216257 0.560860   
guide_right  0.145724 0.306112                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       0.219304 0.201617 0.562886   
guide_top    0.140623 0.328129                              0.000000 0.000000 0.000000                      0.141759 0.141759 0.141759                       0.232811 0.215885 0.606424   

             solution_residual_exact_body_arcsec                    
                                            mean   median      max  
detector_id                                                         
guide_bottom                            0.397042 0.396963 0.398880  
guide_left                              0.414683 0.414706 0.415025  
guide_right                             0.415165 0.415167 0.415264  
guide_top                               0.404762 0.404693 0.406283